# Team Final Overview
이 노트북은 `team_final` 폴더 안에 정리된 최종 실험 구조를 설명하기 위한 요약 노트북입니다.


## 폴더 구조
- `01_k2_select1_baseline`: `k=2`, 군집당 대표 월 `1개`, baseline 평가
- `02_k3_select1_baseline`: `k=3`, 군집당 대표 월 `1개`, baseline 평가
- `03_k4_select1_baseline`: `k=4`, 군집당 대표 월 `1개`, baseline 평가
- `04_k4_select2_model_compare`: `k=4`, 군집당 대표 월 `2개`, 모델 비교 포함


## 파일 구성
각 폴더에는 총 `7개` 파일이 있습니다.
- 개별 스테이션 노트북 `6개`
- `all_6stations_combined_baseline.ipynb` `1개`

개별 스테이션 대상은 다음과 같습니다.
- `ST-481` 상현
- `ST-2425` 다원
- `ST-1331` 찬솔
- `ST-454` 신영
- `ST-453` 혜전
- `ST-2264` 광태


## 진행 흐름
1. 각 스테이션 `2024` 데이터 로드 및 전처리
2. 월별 시간대 패턴 생성
3. 군집 수(`k`)에 따라 월 군집화 수행
4. 각 군집에서 대표 월 선택
5. 선택된 월을 `train 70% / valid 30%`로 분리
6. `2025` 전체 데이터를 외부 테스트로 사용
7. baseline 점수 또는 모델 비교 결과 확인


## 해석 기준
- baseline 폴더 3개는 `HistGradientBoostingRegressor` 기준 점수 확인용입니다.
- `04_k4_select2_model_compare` 폴더는 대표 월을 더 넓게 가져온 뒤, 다른 모델 비교까지 진행하는 확장 실험용입니다.
- 6개 스테이션 통합 파일은 폴더별로 같이 두었지만, 군집 기반 대표 월 선택이 아니라 전체 baseline 비교용입니다.


## 데이터와 타깃 정리
이번 분석은 각 스테이션의 `2024` 데이터를 학습용으로 사용하고, `2025` 전체 데이터를 외부 테스트셋으로 사용합니다.

### 원본 데이터 처리 방식
- 원본 CSV에서 `종료_대여소_ID == 'X'` 인 행은 제거합니다.
- `timestamp` 기준으로 시간대별 집계를 수행합니다.
- 각 시간대에 대해 `inflow`, `outflow`, `total_flow`, `net_flow` 를 생성합니다.

### 주요 타깃 정의
- `inflow`: 해당 시간대 유입량
- `outflow`: 해당 시간대 유출량
- `total_flow`: 해당 시간대 총 이용량 (`inflow + outflow`)
- `net_flow`: 해당 시간대 순이용량 (`inflow - outflow`)

실제 점수 비교의 중심 타깃은 주로 `total_flow` 이며, 일부 baseline 검증에서는 `inflow`, `outflow` 도 함께 확인합니다.


## 사용 피처 정리
현재 노트북들에서 핵심적으로 사용하는 피처는 시간/계절성과 현재 시점 날씨 정보입니다.

### 모델 입력 피처
- `온도`
- `습도`
- `강수량`
- `snow_flag`
- `is_restingday`
- `month_sin`
- `month_cos`
- `hour_sin`
- `hour_cos`

### 피처 사용 원칙
- `lag` 피처와 `rolling` 피처는 사용하지 않습니다.
- 이전 시점 날씨값도 제외하고 현재 시점 정보만 사용합니다.
- 선형성 해석보다 시간대 수요 패턴 반영을 우선하므로 `sin/cos` 형태의 주기 피처를 유지합니다.
- 6개 스테이션 통합 파일에서는 아래 정적 입지 피처도 함께 사용합니다.
  - `위도`, `경도`
  - `residential_index`, `business_index`, `tourism_index`
  - `transit_index`, `commute_in_index`, `commute_out_index`
  - `station_id` 원-핫 인코딩


## 피처별 정보
아래는 현재 분석에서 사용한 주요 피처들의 의미를 정리한 내용입니다.

### 수요 타깃 관련 변수
- `inflow`: 해당 시간대에 스테이션으로 들어온 자전거 수
- `outflow`: 해당 시간대에 스테이션에서 나간 자전거 수
- `total_flow`: 해당 시간대 총 이용량 (`inflow + outflow`)
- `net_flow`: 해당 시간대 순이용량 (`inflow - outflow`)

### 날씨 피처
- `온도`: 해당 시간대 기온
- `습도`: 해당 시간대 습도
- `강수량`: 해당 시간대 강수량
- `snow_flag`: 해당 시간대 적설 또는 눈 관련 여부를 나타내는 플래그 변수

### 시간/계절성 피처
- `hour_sin`: 시간대(0~23)의 주기성을 사인 함수로 변환한 값
- `hour_cos`: 시간대(0~23)의 주기성을 코사인 함수로 변환한 값
- `month_sin`: 월(1~12)의 계절성을 사인 함수로 변환한 값
- `month_cos`: 월(1~12)의 계절성을 코사인 함수로 변환한 값
- `is_restingday`: 주말 또는 휴일 여부를 나타내는 변수

### 원본 시간 파생 변수
- `timestamp`: 기준 시각
- `시간대`: 시 단위 시간 정보
- `month`: 월 정보
- `week_of_month`: 월 내부 주차 정보
- `month_label`: `YYYY-MM` 형태의 월 라벨

### 스테이션 정적 피처
- `위도`: 스테이션 위도
- `경도`: 스테이션 경도
- `residential_index`: 주거지 밀집 정도를 나타내는 지표
- `business_index`: 업무지구 성격을 나타내는 지표
- `tourism_index`: 관광/유동인구 성격을 나타내는 지표
- `transit_index`: 대중교통 접근성과 관련된 지표
- `commute_in_index`: 출근 방향 유입 특성과 관련된 지표
- `commute_out_index`: 퇴근 방향 유출 특성과 관련된 지표
- `station_id`: 스테이션 식별자, 통합 모델에서는 원-핫 인코딩으로 사용

### 분석에서 제외한 피처
- `lag` 피처: 이전 시점 수요값을 사용하는 변수
- `rolling` 피처: 이동평균 등 과거 구간 집계 변수
- 이전 시점 날씨 변수: 예를 들어 `temp_lag_1hr`

이 제외 피처들은 이번 실험에서 해석 일관성과 현재 시점 예측 조건을 유지하기 위해 사용하지 않았습니다.


## 분석 과정 정리
전체 분석은 아래 순서로 진행됩니다.

### 1. 스테이션별 데이터 로드
- 각 스테이션의 `2024` 파일과 `2025` 파일을 불러옵니다.
- 결측 또는 불완전한 대여소 식별값(`X`)은 제거합니다.

### 2. 시간대별 집계 테이블 생성
- `timestamp` 를 기준으로 1시간 1행 형태의 테이블을 만듭니다.
- `inflow`, `outflow`, `total_flow`, `net_flow` 를 계산합니다.
- 시간/월/요일/날씨 관련 피처를 그대로 유지합니다.

### 3. 월별 패턴 생성
- 월별로 `시간대(0~23)` 평균 `total_flow` 를 계산합니다.
- 각 월을 길이 24의 벡터로 표현해 월별 시간대 패턴을 비교합니다.

### 4. 월 군집화
- `AgglomerativeClustering` 으로 월 패턴을 군집화합니다.
- 시나리오에 따라 `k=2`, `k=3`, `k=4` 를 사용합니다.
- 각 군집에서 중심 패턴에 가까운 월을 대표 월로 선택합니다.

### 5. 대표 월 선택
- baseline 시나리오에서는 군집당 대표 월 `1개`를 선택합니다.
- 확장 시나리오에서는 군집당 대표 월 `2개`를 선택합니다.
- 선택된 월들만 사용해 학습/검증용 데이터를 만듭니다.

### 6. 학습/검증 분리
- 선택된 각 월 내부를 시간순 `7:3` 으로 분리합니다.
- 앞 `70%` 는 `train`, 뒤 `30%` 는 `valid` 로 둡니다.

### 7. 외부 테스트 구성
- `2025` 전체 데이터를 외부 테스트셋으로 사용합니다.
- 이 단계는 연도 변화가 있는 실제 일반화 성능을 확인하기 위한 목적입니다.

### 8. 점수 계산
- `R²`, `MAE`, `RMSE` 를 `train / valid / test` 에 대해 계산합니다.
- 필요하면 실제값과 예측값 시계열 그래프도 함께 확인합니다.


## 폴더별 의미
각 폴더는 군집 수와 대표 월 선택 수에 따라 나뉩니다.

### `01_k2_select1_baseline`
- `k=2`
- 군집당 대표 월 `1개`
- baseline 점수 확인용

### `02_k3_select1_baseline`
- `k=3`
- 군집당 대표 월 `1개`
- baseline 점수 확인용

### `03_k4_select1_baseline`
- `k=4`
- 군집당 대표 월 `1개`
- baseline 점수 확인용

### `04_k4_select2_model_compare`
- `k=4`
- 군집당 대표 월 `2개`
- 대표 월을 더 넓게 가져온 실험용
- 현재는 개별 파일 구조를 짧게 정리해 baseline 형태로 통일해 둔 상태입니다.


## 점수 해석 가이드
점수는 단순히 높고 낮음만 보는 것이 아니라 `train`, `valid`, `test` 간 차이도 함께 봐야 합니다.

### 해석 포인트
- `train` 이 매우 높고 `valid / test` 가 낮으면 과적합 가능성이 큽니다.
- `valid` 와 `test` 가 함께 안정적이면 대표 월 선택이 비교적 잘 된 것입니다.
- `test R²` 는 외부 일반화 성능의 핵심 기준입니다.
- `MAE`, `RMSE` 는 실제 예측 오차 크기를 함께 보여줍니다.

### 실무적 판단 기준
- 단일 스테이션 실험은 대표 월 선택 전략의 타당성을 보기 위한 목적이 큽니다.
- 6개 스테이션 통합 파일은 전체 baseline 비교용으로 해석합니다.
- 군집 수를 늘리거나 대표 월 수를 늘리는 이유는 `train` 과 `test` 사이의 격차를 줄이기 위해서입니다.
